# Two-Stage Adaptive Conformal — S&P 500 Stocks (NBEATS)

Adaptive conformal prediction on stock prices with a two-stage forecaster:
stage 1 is an NBEATS model forecasting AAPL/AMZN/MSFT, stage 2 an OLS that
maps those to the AAPL target. Adaptive methods come from the shared
`adaptive_conformal.py`.

1. Load `stocks/all_stocks_5yr.csv`, train NBEATS, fit the stage-2 OLS.
2. Roll forward to build the split-residual conformal scores (r, r1, r2).
3. Run the adaptive methods (FWER, ACI, PID, OCID, ECI) and plot width /
   coverage / per-method intervals.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from darts import TimeSeries
from darts.models import NBEATSModel
from darts.dataprocessing.transformers import Scaler

from adaptive_conformal import (
    adaptive_fwer_alpha, aci_clipped, quantile_integrator_log_scorecaster,
    decay_quantile, ECI, sliding_window_average,
)

## 1. Load data, train NBEATS (stage 1), fit OLS (stage 2)

In [ ]:
indicator_df = pd.read_csv("stocks/all_stocks_5yr.csv", parse_dates=["date"])
wide = indicator_df.pivot(index="date", columns="Name", values="close")
wide = wide.reset_index(drop=False)
dates = wide["date"]
data = wide.drop(columns=["date"])

In [ ]:
series_all = TimeSeries.from_dataframe(data)

In [ ]:
train, val = series_all.split_after(600)
other_val = val[["AAPL","AMZN", "MSFT"]]

In [ ]:
other_train = train[["AAPL","AMZN", "MSFT"]]
scaler = Scaler()
other_train_scaled = scaler.fit_transform(other_train)
other_val = scaler.fit_transform(other_val)

In [ ]:
model = NBEATSModel(
    input_chunk_length=24,
    output_chunk_length = 6,
    dropout=0.3,
    batch_size=32,
    n_epochs=100,
    optimizer_kwargs={'lr': 1e-4},
    random_state=42
)

model.fit(
    series=other_train_scaled,
    verbose  = True
)

In [ ]:
other_val_temp = other_val[:300]
other_val_temp_aapl = other_val_temp['AAPL']

conformal_set = other_val[270:]
conformal_set_aapl = conformal_set["AAPL"]

In [ ]:
forecast_horizon = 6
window_size = 24
step_size = 1

preds = []
targets = []
feats = []
for i in range(0, len(other_val_temp) - forecast_horizon - window_size, step_size):
    input_series = other_val[i:i + window_size]
    feats.append(input_series.values()[-1,:])

    forecast = model.predict(
        n=forecast_horizon,
        series=input_series,
        verbose=False
    )

    preds.append(forecast.values()[-1])
    targets.append(other_val_temp_aapl[i+window_size+forecast_horizon].values())

In [ ]:
targets = np.asarray(targets).flatten()
preds = np.asarray(preds)
feats = np.asarray(feats)

In [ ]:
combined_with_lag = np.concatenate((preds, feats), axis=1)

In [ ]:
plt.plot(preds[:,0])
plt.plot(other_val_temp.values()[window_size+forecast_horizon:,0])

In [ ]:
ols = sm.OLS(targets, preds)
res = ols.fit_regularized(alpha = 0.001, L1_wt = 0)

In [ ]:
plt.plot(res.predict(preds))
plt.plot(targets)

## 2. Build split-residual conformal scores

Rolling NBEATS forecast → stage-2 OLS prediction; calculate residual/components `r` is the end-to-end
residual, `r2` the stage-2-only residual, `r1 = |r − r2|`.

In [ ]:
aapl_preds = []
conformal_r1_score = []
conformal_r2_score = []
conformal_scores = []

for i in range(0, len(conformal_set) - forecast_horizon - window_size, step_size):
    input_series = conformal_set[i:i + window_size]

    forecast = model.predict(
        n=forecast_horizon,
        series=input_series,
        verbose=False
    )

    aapl_pred = res.predict(forecast.values()[-1])
    aapl_preds.append(aapl_pred)
    #the 30 is to correct for window size and forecast horizon
    full_resid = np.abs(aapl_pred - conformal_set_aapl[i+30].values()[0])
    conformal_r2 = np.abs(res.predict(conformal_set[i+30].values()[0])-conformal_set_aapl[i+30].values()[0])
    conformal_r1 = np.abs(full_resid - conformal_r2)
    conformal_r1_score.append(conformal_r1)
    conformal_r2_score.append(conformal_r2)
    conformal_scores.append(full_resid)

conformal_r1_score = np.asarray(conformal_r1_score)
conformal_r2_score = np.asarray(conformal_r2_score)
conformal_scores = np.asarray(conformal_scores)

## 3. Adaptive methods + figures

In [ ]:
alpha = 0.1
epsilon = 0.01
window_length = 100
T_burnin = 10
lambda_set = grid = np.linspace(0.3, 1, 4)
lambda_a,lambda_b = np.meshgrid(grid,grid)
lambda_set = np.column_stack((lambda_a.flatten(), lambda_b.flatten()))
delta = 0.9
lr = 0.01
alpha_lr = 0.01

eps_us = 0.1
etas_us_decaying = np.array([100/(t**(1/2+eps_us)) for t in range(1, len(conformal_scores)+1)])

results = adaptive_fwer_alpha(
        conformal_scores,
        alpha,
        epsilon,
        window_length,
        window_length,
        T_burnin,
        lambda_set,
        conformal_r1_score,
        conformal_r2_score,
        delta,
        None,
        forecast_horizon,
        lr,
        alpha_lr)



res_aci = aci_clipped(conformal_scores,
alpha,
alpha_lr,
window_length,
T_burnin,
forecast_horizon)

train_size = 600

Csat = 2/np.pi*(np.ceil(np.log(train_size-T_burnin)*0.2) - 1/np.log(train_size-T_burnin))
KI = .6

res_pid = quantile_integrator_log_scorecaster(
    conformal_scores,
    alpha,
    alpha_lr,
    conformal_set,
    T_burnin,
    Csat,
    KI,
    'upper',
    forecast_horizon,
    integrate=True,
    proportional_lr=True,
    scorecast=False,
    index_forecast=True,
    )

res_pid_sc = quantile_integrator_log_scorecaster(
    conformal_scores,
    alpha,
    alpha_lr,
    conformal_set,
    T_burnin,
    Csat,
    KI,
    'upper',
    forecast_horizon,
    integrate=True,
    proportional_lr=True,
    scorecast=True,
    index_forecast=True,
    )

eps = 0.1
etas_decaying = np.array([1/(t**(1/2+eps)) for t in range(1, len(conformal_scores)+1)])
q_1 = conformal_scores[0]
res_decay = decay_quantile(conformal_scores, q_1, etas_decaying, alpha, forecast_horizon)

res_eci = ECI(conformal_scores,
    alpha,
    alpha_lr,
    T_burnin,
    forecast_horizon)

In [ ]:
sliding_window_len = 100
plt.style.use('default')
plt.plot(dates.iloc[900+window_length+T_burnin+1:],results['q'][window_length+T_burnin:], label="Split Residuals", linestyle="-", color="C0")
plt.plot(dates.iloc[900+window_length+T_burnin+1:],res_aci['q'][window_length+T_burnin:], label="ACI (clipped)", linestyle="--", color="C1")
plt.plot(dates.iloc[900+window_length+T_burnin+1:],res_pid_sc['q'][window_length+T_burnin:], label="PID", linestyle="-.", color="C2")
plt.plot(dates.iloc[900+window_length+T_burnin+1:],res_decay['q'][window_length+T_burnin:], label="OCID", linestyle="-.", color="C3")
plt.plot(dates.iloc[900+window_length+T_burnin+1:],res_eci['q'][window_length+T_burnin:], label="ECI", linestyle="--", color="C4")
plt.xlabel("Date")

plt.ylabel("Prediction Interval Width")
plt.legend(loc="upper right", frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig(f"pi_stock_width_{i}.png", dpi=300, bbox_inches='tight')

plt.show()

array_cov_us = sliding_window_average(results['cov'][T_burnin+window_length -30:], sliding_window_len)
plt.plot(dates.iloc[870+ sliding_window_len+window_length+T_burnin:],array_cov_us,linestyle="-", color = 'C0', label= 'Split Residuals')

array_cov_aci = sliding_window_average(res_aci['cov'][T_burnin+window_length -30:], sliding_window_len)

plt.plot(dates.iloc[870+ sliding_window_len+window_length+T_burnin:],array_cov_aci,linestyle="--", color = 'C1', label = 'ACI (clipped)')

array_cov_pid = sliding_window_average(res_pid['cov'][T_burnin+window_length -30:], sliding_window_len)

plt.plot(dates.iloc[870+ sliding_window_len+window_length+T_burnin:],array_cov_pid, linestyle="-.",color = 'C2', label='PID')

array_cov_decay = sliding_window_average(res_decay['cov'][T_burnin+window_length -30:], sliding_window_len)
plt.plot(dates.iloc[870+ sliding_window_len+window_length+T_burnin:],array_cov_decay, linestyle="-.",color = 'C3', label='OCID')

array_cov_eci = sliding_window_average(res_eci['cov'][T_burnin+window_length -30:], sliding_window_len)

plt.plot(dates.iloc[870+ sliding_window_len+window_length+T_burnin:],array_cov_eci,linestyle="--", color = 'C4', label = 'ECI')

plt.axhline(0.9, color='gray', linestyle='--', linewidth=1, label='90% Coverage')
plt.xlabel("Date")

plt.ylabel("Average Coverage (100 Sliding Window)")
plt.ylim(0.45, 1.01)
plt.legend(loc="lower right", frameon=False, fontsize=8)
plt.savefig(f"pi_stock_cov_{i}.png", dpi=300, bbox_inches='tight')
plt.show()

plt.fill_between(dates.iloc[900+window_length+window_size+1:],np.asarray(aapl_preds[window_length+window_size:]).flatten()- results['q'][window_length+window_size:],np.asarray(aapl_preds[window_length+window_size:]).flatten()+ results['q'][window_length+window_size:], label="Split Residual Interval", color="C0",alpha=0.2)
plt.plot(dates.iloc[900+window_length+window_size+1:], aapl_preds[window_length+window_size:], label ='Predicted Value', color = 'C6')
plt.plot(dates.iloc[900+window_length+window_size+1:],conformal_set_aapl[window_length+window_size+ 30:].values(), label = 'True Value', color = 'C7')
plt.xlabel("Date")
plt.ylabel("AAPL Index")

plt.legend(loc="upper left", frameon=False, fontsize=10)

plt.savefig(f"us_actual_stock_{i}.png", dpi=300, bbox_inches='tight')
plt.show()
plt.fill_between(dates.iloc[900+window_length+window_size+1:],np.asarray(aapl_preds[window_length+window_size:]).flatten()- res_aci['q'][window_length+window_size:],np.asarray(aapl_preds[window_length+window_size:]).flatten()+ res_aci['q'][window_length+window_size:], label="ACI (Clipped) Interval", color="C1",alpha=0.2)
plt.plot(dates.iloc[900+window_length+window_size+1:],aapl_preds[window_length+window_size:], label ='Predicted Value', color = 'C6')
plt.plot(dates.iloc[900+window_length+window_size+1:],conformal_set_aapl[window_length+window_size+ 30:].values(), label = 'True Value', color = 'C7')
plt.xlabel("Date")
plt.ylabel("AAPL Index")

plt.legend(loc="upper left", frameon=False, fontsize=10)

plt.savefig(f"aci_actual_stock_{i}.png", dpi=300, bbox_inches='tight')
plt.show()
plt.fill_between(dates.iloc[900+window_length+window_size+1:],np.asarray(aapl_preds[window_length+window_size:]).flatten()- res_pid['q'][window_length+window_size:],np.asarray(aapl_preds[window_length+window_size:]).flatten()+ res_pid['q'][window_length+window_size:], label="PID Interval", color="C2",alpha=0.2)
plt.plot(dates.iloc[900+window_length+window_size+1:],aapl_preds[window_length+window_size:], label ='Predicted Value', color = 'C6')
plt.plot(dates.iloc[900+window_length+window_size+1:], conformal_set_aapl[window_length+window_size + 30:].values(), label = 'True Value', color = 'C7')
plt.xlabel("Date")
plt.ylabel("AAPL Index")

plt.legend(loc="upper left", frameon=False, fontsize=10)

plt.savefig(f"pid_actual_stock_{i}.png", dpi=300, bbox_inches='tight')
plt.show()

plt.fill_between(dates.iloc[900+window_length+window_size+1:],np.asarray(aapl_preds[window_length+window_size:]).flatten()- res_decay['q'][window_length+window_size:],np.asarray(aapl_preds[window_length+window_size:]).flatten()+ res_decay['q'][window_length+window_size:], label="OCID Interval", color="C3",alpha=0.2)
plt.plot(dates.iloc[900+window_length+window_size+1:],aapl_preds[window_length+window_size:], label ='Predicted Value', color = 'C6')
plt.plot(dates.iloc[900+window_length+window_size+1:],conformal_set_aapl[window_length+window_size+ 30:].values(), label = 'True Value', color = 'C7')
plt.xlabel("Date")
plt.ylabel("AAPL Index")

plt.legend(loc="upper left", frameon=False, fontsize=10)

plt.savefig(f"decay_actual_stock_{i}.png", dpi=300, bbox_inches='tight')
plt.show()

plt.fill_between(dates.iloc[900+window_length+window_size+1:],np.asarray(aapl_preds[window_length+window_size:]).flatten()- res_eci['q'][window_length+window_size:],np.asarray(aapl_preds[window_length+window_size:]).flatten()+ res_eci['q'][window_length+window_size:], label="ECI Interval", color="C4",alpha=0.2)
plt.plot(dates.iloc[900+window_length+window_size+1:],aapl_preds[window_length+window_size:], label ='Predicted Value', color = 'C6')
plt.plot(dates.iloc[900+window_length+window_size+1:],conformal_set_aapl[window_length+window_size+ 30:].values(), label = 'True Value', color = 'C7')
plt.xlabel("Date")
plt.ylabel("AAPL Index")

plt.legend(loc="upper left", frameon=False, fontsize=10)

plt.savefig(f"eci_actual_stock_{i}.png", dpi=300, bbox_inches='tight')
plt.show()